# BÀI TẬP 2: HỌC CÓ GIÁM SÁT (Máy học – Ts. Nguyễn An Tế)
1. kNN tự cài đặt (Iris) · 2. kNN trên Wine · 3–4. Cây quyết định, cắt tỉa CCP (Breast Cancer) · 5. Naive Bayes · 6. SVM trên Moons · 7. Naive Bayes trên Circles · 8. SVM Breast Cancer (C, gamma, GridSearch) · 9. OOP

In [ ]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_moons, make_circles
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
warnings.filterwarnings('ignore'); np.random.seed(0)

---
## 1. kNN tự cài đặt (không dùng `KNeighborsClassifier`) trên Iris
**Ý tưởng:** lười học (lazy) – `fit` chỉ lưu dữ liệu; `predict` tính khoảng cách tới mọi mẫu train, lấy k láng giềng gần nhất rồi bỏ phiếu.

**KHI NÀO DÙNG:** dữ liệu nhỏ/vừa, ranh giới lớp phức tạp, features cùng thang đo (hoặc đã chuẩn hóa). *Không hợp* khi dữ liệu rất lớn (predict chậm) hoặc số chiều quá cao (lời nguyền chiều).

In [ ]:
class KNN:
    """kNN tự cài đặt: k, khoảng cách (euclidean/manhattan/chebyshev), trọng số (uniform/distance)."""
    def __init__(self, k=3, metric='euclidean', weights='uniform'):
        self.k, self.metric, self.weights = k, metric, weights

    def fit(self, X, y):
        self.X_ = np.asarray(X, float); self.y_ = np.asarray(y); self.classes_ = np.unique(self.y_); return self

    def _dist(self, x):
        d = self.X_ - x
        if self.metric == 'euclidean': return np.sqrt((d ** 2).sum(1))
        if self.metric == 'manhattan': return np.abs(d).sum(1)
        if self.metric == 'chebyshev': return np.abs(d).max(1)
        raise ValueError('metric không hợp lệ')

    def _predict_one(self, x):
        d = self._dist(x); idx = np.argsort(d)[:self.k]
        if self.weights == 'uniform':
            return Counter(self.y_[idx]).most_common(1)[0][0]
        w = 1 / (d[idx] + 1e-9)                        # láng giềng càng gần càng có trọng số lớn
        phieu = {c: w[self.y_[idx] == c].sum() for c in self.classes_}
        return max(phieu, key=phieu.get)

    def predict(self, X): return np.array([self._predict_one(x) for x in np.asarray(X, float)])
    def score(self, X, y): return np.mean(self.predict(X) == np.asarray(y))
    def error(self, X, y): return 1 - self.score(X, y)

iris = load_iris(as_frame=True); Xi, yi = iris.data, iris.target
ten_hoa = iris.target_names
print(Xi.shape, ten_hoa)

### 1.1–1.2 Xây dựng với k xác định trước, đánh giá trên **tập huấn luyện**

In [ ]:
knn = KNN(k=5).fit(Xi, yi)
print('Accuracy (train) = %.4f | Error = %.4f' % (knn.score(Xi, yi), knn.error(Xi, yi)))
for k in [1, 3, 5, 9, 15]:
    print(f'k={k:2d}: accuracy train = {KNN(k).fit(Xi, yi).score(Xi, yi):.4f}')

> Lưu ý: với k=1 accuracy train = 100% vì mỗi mẫu chính là láng giềng gần nhất của nó → **không phản ánh khả năng tổng quát**; cần tập test (câu 2).

In [ ]:
## 1.3 Thử nghiệm nhiều lần để dự đoán loài hoa
rng = np.random.default_rng(0)
for _ in range(5):
    mau = Xi.values[rng.integers(len(Xi))] + rng.normal(0, 0.2, 4)   # mẫu mới = mẫu thật + nhiễu nhỏ
    print(np.round(mau, 2), '→', ten_hoa[knn.predict([mau])[0]])
print('\nMột mẫu tự nhập:', ten_hoa[knn.predict([[5.9, 3.0, 5.1, 1.8]])[0]])

### 1.4 Mở rộng: trọng số cho láng giềng (weights = 1/distance)

In [ ]:
print('k | uniform | distance')
for k in [1, 3, 5, 9, 15, 30]:
    a = KNN(k, weights='uniform').fit(Xi, yi).score(Xi, yi); b = KNN(k, weights='distance').fit(Xi, yi).score(Xi, yi)
    print(f'{k:2d} | {a:.4f}  | {b:.4f}')

> Khi k lớn, `distance` giữ cho các láng giềng gần có ảnh hưởng nhiều hơn → ít bị kéo về lớp đa số ở xa. (Accuracy train của `distance` luôn = 1 vì mẫu trùng chính nó có khoảng cách ≈ 0 → trọng số rất lớn; chỉ so sánh có ý nghĩa trên tập test.)

---
## 2. kNN trên Wine (80:20): siêu tham số k và khoảng cách
Wine có các features khác thang đo (Proline ~ hàng trăm–nghìn) nên so sánh **cả khi chưa và đã chuẩn hóa**.
**KHI NÀO CHUẨN HÓA:** mọi thuật toán dựa trên khoảng cách (kNN, SVM, k-Means) → bắt buộc nếu thang đo features khác nhau.

In [ ]:
wine = load_wine(as_frame=True); Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=1, stratify=yw)
sc = StandardScaler().fit(Xw_tr); Xw_tr_s, Xw_te_s = sc.transform(Xw_tr), sc.transform(Xw_te)

ks = list(range(1, 26, 2)); dists = ['euclidean', 'manhattan', 'chebyshev']
def quet(Xtr, Xte):
    return pd.DataFrame({d: [KNN(k, d).fit(Xtr, yw_tr).score(Xte, yw_te) for k in ks] for d in dists}, index=ks)
ket_qua = {'Chưa chuẩn hóa': quet(Xw_tr.values, Xw_te.values), 'Đã chuẩn hóa': quet(Xw_tr_s, Xw_te_s)}
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (ten, df) in zip(ax, ket_qua.items()):
    df.plot(ax=a, marker='o'); a.set_title(ten); a.set_xlabel('k'); a.set_ylabel('Accuracy (test)')
plt.tight_layout(); plt.show()
for ten, df in ket_qua.items(): print(ten, '\n', df.max().round(4).to_dict(), '| k tốt nhất:', df.idxmax().to_dict())

---
## 3. Cây quyết định (Gini) trên Breast Cancer
**KHI NÀO DÙNG:** cần mô hình **dễ giải thích** (luật if–then), dữ liệu hỗn hợp, không cần chuẩn hóa. *Nhược điểm:* không giới hạn độ sâu → overfit.

In [ ]:
bc = load_breast_cancer(as_frame=True); Xb, yb = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.2, random_state=1, stratify=yb)

cay = DecisionTreeClassifier(criterion='gini', random_state=1).fit(Xb_tr, yb_tr)
print('Accuracy train = %.4f | test = %.4f | độ sâu = %d | số lá = %d' % (cay.score(Xb_tr, yb_tr), cay.score(Xb_te, yb_te), cay.get_depth(), cay.get_n_leaves()))

cay3 = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=1).fit(Xb_tr, yb_tr)   # sơ đồ gọn cho dễ nhìn
plt.figure(figsize=(18, 8)); plot_tree(cay3, feature_names=bc.feature_names, class_names=bc.target_names, filled=True, fontsize=9)
plt.title('Cây quyết định (Gini, max_depth=3 để dễ đọc)'); plt.show()

### 3.3 Mức độ quan trọng của features (`feature_importances_`)

In [ ]:
imp = pd.Series(cay.feature_importances_, index=bc.feature_names).sort_values(ascending=False)
print(imp[imp > 0].round(4).to_string())
imp.head(10)[::-1].plot.barh(figsize=(7, 4), title='Top 10 feature importance'); plt.show()

---
## 4. Độ sâu và Cost Complexity Pruning (CCP)
**Đọc kết quả:** độ sâu tăng → train tiến tới 100% còn test chững/giảm (overfit). Cắt tỉa theo α: α nhỏ → cây to (overfit), α lớn → cây quá nhỏ (underfit). **Chọn α** tại điểm test cao nhất mà khoảng cách train–test còn nhỏ (hoặc dùng cross-validation).

In [ ]:
## 4.2 Hiệu quả theo độ sâu
sau = range(1, 16); tr, te = [], []
for d in sau:
    m = DecisionTreeClassifier(max_depth=d, random_state=1).fit(Xb_tr, yb_tr); tr.append(m.score(Xb_tr, yb_tr)); te.append(m.score(Xb_te, yb_te))
plt.plot(sau, tr, 'o-', label='Train'); plt.plot(sau, te, 's-', label='Test'); plt.xlabel('max_depth'); plt.ylabel('Accuracy'); plt.legend(); plt.title('Accuracy theo độ sâu'); plt.show()

## 4.3 Các giá trị ccp_alpha
path = DecisionTreeClassifier(random_state=1).cost_complexity_pruning_path(Xb_tr, yb_tr)
alphas = path.ccp_alphas[:-1]            # bỏ alpha cuối (cây chỉ còn nút gốc)
print('Số giá trị α:', len(alphas)); print(np.round(alphas[:10], 5))

In [ ]:
## 4.4 Accuracy theo α (Train vs Test)
cays = [DecisionTreeClassifier(ccp_alpha=a, random_state=1).fit(Xb_tr, yb_tr) for a in alphas]
tr = [c.score(Xb_tr, yb_tr) for c in cays]; te = [c.score(Xb_te, yb_te) for c in cays]
plt.figure(figsize=(8, 4)); plt.plot(alphas, tr, marker='o', drawstyle='steps-post', label='Train')
plt.plot(alphas, te, marker='s', drawstyle='steps-post', label='Test')
plt.xlabel('ccp_alpha'); plt.ylabel('Accuracy'); plt.title('Accuracy theo α'); plt.legend(); plt.show()
gap = np.array(tr) - np.array(te); i_best = int(np.argmax(np.array(te) - 0.5 * np.abs(gap)))
print('α gợi ý = %.5f | train = %.4f | test = %.4f | số lá = %d' % (alphas[i_best], tr[i_best], te[i_best], cays[i_best].get_n_leaves()))

---
## 5. Naive Bayes trên Breast Cancer
| Mô hình | Giả định về feature | KHI NÀO DÙNG |
|---|---|---|
| GaussianNB | liên tục, xấp xỉ chuẩn trong từng lớp | Features số thực (dạng chuông) |
| BernoulliNB | nhị phân 0/1 | Có/không (sự hiện diện); với dữ liệu liên tục phải **nhị phân hóa** trước |
| MultinomialNB | đếm/tần suất **không âm** | Văn bản (đếm từ), dữ liệu đếm; không dùng được khi có giá trị âm |

In [ ]:
## 5.1 Phân phối của các features
bc_df = bc.data.copy(); bc_df['target'] = bc.target
fig, axes = plt.subplots(6, 5, figsize=(18, 14))
for a, c in zip(axes.ravel(), bc.feature_names):
    for t, nm in enumerate(bc.target_names):
        a.hist(bc_df.loc[bc_df.target == t, c], bins=20, alpha=.5, label=nm)
    a.set_title(c, fontsize=8); a.tick_params(labelsize=6)
axes[0, 0].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
class NaiveBayes_Experiment:
    """So sánh GaussianNB, BernoulliNB, MultinomialNB trên cùng dữ liệu."""
    def __init__(self, X_tr, X_te, y_tr, y_te):
        self.X_tr, self.X_te, self.y_tr, self.y_te = X_tr, X_te, y_tr, y_te

    def run(self):
        Xtr, Xte = np.asarray(self.X_tr, float), np.asarray(self.X_te, float)
        nguong = np.median(Xtr, axis=0)                         # nhị phân hóa theo trung vị của mỗi feature (fit trên train)
        mn = MinMaxScaler().fit(Xtr)                            # đưa về không âm (≥ 0) cho Multinomial
        cac_mo_hinh = {'GaussianNB': (GaussianNB(), Xtr, Xte),
                       'BernoulliNB (nhị phân hóa)': (BernoulliNB(binarize=None), (Xtr > nguong).astype(int), (Xte > nguong).astype(int)),
                       'MultinomialNB (MinMax)': (MultinomialNB(), mn.transform(Xtr), mn.transform(Xte))}
        self.models, rows = {}, []
        for ten, (m, a, b) in cac_mo_hinh.items():
            m.fit(a, self.y_tr); self.models[ten] = (m, b)
            rows.append({'Mô hình': ten, 'Acc_train': m.score(a, self.y_tr), 'Acc_test': m.score(b, self.y_te)})
        self.ket_qua = pd.DataFrame(rows).set_index('Mô hình'); return self.ket_qua

    def chi_tiet(self):
        for ten, (m, b) in self.models.items():
            print('===', ten); print(confusion_matrix(self.y_te, m.predict(b)))
            print(classification_report(self.y_te, m.predict(b), digits=3))

nb_bc = NaiveBayes_Experiment(Xb_tr, Xb_te, yb_tr, yb_te); print(nb_bc.run().round(4)); nb_bc.chi_tiet()

**Nhận xét:** GaussianNB thường tốt nhất vì features liên tục dạng chuông. BernoulliNB mất thông tin do nhị phân hóa; MultinomialNB không đúng bản chất dữ liệu (không phải dữ liệu đếm). Chọn mô hình theo **kiểu dữ liệu**, không chỉ theo accuracy.

---
## 6. SVM trên Moons: decision boundary
**KHI NÀO DÙNG kernel:** `linear` khi dữ liệu tách được bằng đường thẳng/ nhiều features hơn mẫu; `rbf` mặc định cho ranh giới cong, phi tuyến; `poly` khi biết tương tác đa thức bậc thấp.

In [ ]:
def ve_bien_quyet_dinh(model, X, y, ax=None, title=''):
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - .5, X[:, 0].max() + .5; y0, y1 = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=.3, cmap='coolwarm'); ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=20)
    ax.set_title(title)

Xm, ym = make_moons(300, noise=0.25, random_state=1)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.2, random_state=1)

class SVM_Experiment:
    def __init__(self, X_tr, X_te, y_tr, y_te): self.X_tr, self.X_te, self.y_tr, self.y_te = X_tr, X_te, y_tr, y_te
    def so_sanh_kernel(self, kernels=('linear', 'rbf', 'poly'), **kw):
        fig, ax = plt.subplots(1, len(kernels), figsize=(5 * len(kernels), 4)); rows = []
        for a, k in zip(np.atleast_1d(ax), kernels):
            m = SVC(kernel=k, **kw).fit(self.X_tr, self.y_tr)
            ve_bien_quyet_dinh(m, self.X_tr, self.y_tr, a, f'{k}: test acc = {m.score(self.X_te, self.y_te):.3f}')
            rows.append({'kernel': k, 'train': m.score(self.X_tr, self.y_tr), 'test': m.score(self.X_te, self.y_te), 'n_SV': int(m.n_support_.sum())})
        plt.tight_layout(); plt.show(); return pd.DataFrame(rows).set_index('kernel')

svm_m = SVM_Experiment(Xm_tr, Xm_te, ym_tr, ym_te)
print(svm_m.so_sanh_kernel())

**Nhận xét:** Moons không tách tuyến tính → `linear` có biên thẳng, sai nhiều; `rbf` và `poly` uốn cong theo hình trăng nên chính xác hơn.

---
## 7. Bài 5 trên bộ dữ liệu Circles (phân phối, GaussianNB/BernoulliNB/MultinomialNB, đánh giá)

In [ ]:
Xc, yc = make_circles(300, noise=0.1, factor=0.5, random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=1)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap='coolwarm', edgecolor='k', s=15); ax[0].set_title('Circles')
for j in range(2):
    for t in [0, 1]: ax[j + 1].hist(Xc[yc == t, j], bins=20, alpha=.5, label=f'lớp {t}')
    ax[j + 1].set_title(f'Phân phối feature {j}'); ax[j + 1].legend()
plt.tight_layout(); plt.show()

nb_c = NaiveBayes_Experiment(Xc_tr, Xc_te, yc_tr, yc_te); print(nb_c.run().round(4))

**Nhận xét:** hai lớp của Circles có **cùng tâm, cùng trung bình** nhưng **phương sai khác nhau** (vòng trong nhỏ, vòng ngoài lớn). GaussianNB ước lượng phương sai riêng cho từng lớp nên tạo được biên cong (dạng elip/tròn) và đạt độ chính xác cao (~97% trong lần chạy thử). Ngược lại BernoulliNB và MultinomialNB chỉ nhìn tần suất/ngưỡng từng feature riêng lẻ, không nắm được quan hệ "gần tâm hay xa tâm" nên chỉ ở mức đoán ngẫu nhiên (~50%). Bài học: chọn biến thể NB theo **kiểu phân phối của feature**; đồng thời SVM-RBF (bên dưới) cũng xử lý tốt vì tạo được biên phi tuyến.

In [ ]:
## Đối chiếu SVM-RBF trên Circles
print(SVM_Experiment(Xc_tr, Xc_te, yc_tr, yc_te).so_sanh_kernel(kernels=('linear', 'rbf', 'poly')))

---
## 8. SVM trên Breast Cancer
SVM nhạy với thang đo → luôn dùng `make_pipeline(StandardScaler(), SVC(...))`.

In [ ]:
## 8.1 Ảnh hưởng của C
C_list = [0.1, 1, 10, 100]; rows = []; n_sv = []
for C_ in C_list:
    m = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=C_, gamma='scale')).fit(Xb_tr, yb_tr)
    s = m[-1]; rows.append({'C': C_, 'train': m.score(Xb_tr, yb_tr), 'test': m.score(Xb_te, yb_te), 'n_support_vectors': int(s.n_support_.sum())})
bang_C = pd.DataFrame(rows).set_index('C'); print(bang_C.round(4))

## 8.2 Số support vectors theo C
bang_C['n_support_vectors'].plot(marker='o', logx=True, figsize=(6, 3.5), title='Số support vectors theo C'); plt.ylabel('#SV'); plt.show()

**Đọc kết quả:** C nhỏ → lề rộng, cho phép sai nhiều (regularization mạnh) → **nhiều support vectors**, mô hình đơn giản (có thể underfit). C lớn → phạt vi phạm nặng, lề hẹp → ít SV hơn, đường biên bám dữ liệu train (nguy cơ overfit).

### 8.3 Ảnh hưởng của gamma (kernel RBF) lên hình dạng biên quyết định
Dữ liệu 30 chiều không vẽ được biên → chiếu xuống **2 thành phần chính (PCA)** rồi huấn luyện SVM trên không gian 2D chỉ để minh họa.

In [ ]:
pca = make_pipeline(StandardScaler(), PCA(2)).fit(Xb_tr); Z_tr, Z_te = pca.transform(Xb_tr), pca.transform(Xb_te)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for a, g in zip(ax, [0.1, 1.0, 100]):
    m = SVC(kernel='rbf', gamma=g, C=1).fit(Z_tr, yb_tr)
    ve_bien_quyet_dinh(m, Z_tr, yb_tr, a, f'gamma={g}: train={m.score(Z_tr, yb_tr):.3f}, test={m.score(Z_te, yb_te):.3f}')
plt.tight_layout(); plt.show()

**Đọc kết quả:** gamma nhỏ → mỗi điểm ảnh hưởng xa, biên **mượt, gần tuyến tính** (underfit). gamma lớn → ảnh hưởng rất cục bộ, biên **lởm chởm, bao quanh từng điểm** (train ≈ 100%, test giảm → overfit).

### 8.4 GridSearch tìm (C, gamma) tối ưu

In [ ]:
luoi = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': [0.001, 0.01, 0.1, 1]}
gs = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel='rbf')), luoi, cv=5, scoring='accuracy').fit(Xb_tr, yb_tr)
print('Tham số tốt nhất:', gs.best_params_, '| CV acc = %.4f | Test acc = %.4f' % (gs.best_score_, gs.score(Xb_te, yb_te)))
heat = pd.DataFrame(gs.cv_results_).pivot(index='param_svc__C', columns='param_svc__gamma', values='mean_test_score').astype(float)
plt.imshow(heat.values, cmap='viridis'); plt.colorbar(label='CV accuracy')
plt.xticks(range(heat.shape[1]), heat.columns); plt.yticks(range(heat.shape[0]), heat.index); plt.xlabel('gamma'); plt.ylabel('C')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]): plt.text(j, i, f'{heat.values[i, j]:.3f}', ha='center', va='center', color='w', fontsize=8)
plt.title('GridSearch (C, gamma)'); plt.show()

---
## 9. Lập trình hướng đối tượng
Các lớp đã dùng: `KNN` (kNN tự cài), `NaiveBayes_Experiment`, `SVM_Experiment`, hàm `ve_bien_quyet_dinh` dùng chung. Cây quyết định dùng trực tiếp `DecisionTreeClassifier`; có thể gói thêm vào lớp như sau:

In [ ]:
class Tree_Experiment:
    def __init__(self, X_tr, X_te, y_tr, y_te): self.X_tr, self.X_te, self.y_tr, self.y_te = X_tr, X_te, y_tr, y_te
    def theo_do_sau(self, sau=range(1, 11)):
        return pd.DataFrame([{'depth': d, 'train': (m := DecisionTreeClassifier(max_depth=d, random_state=1).fit(self.X_tr, self.y_tr)).score(self.X_tr, self.y_tr),
                              'test': m.score(self.X_te, self.y_te)} for d in sau]).set_index('depth')
    def theo_alpha(self):
        al = DecisionTreeClassifier(random_state=1).cost_complexity_pruning_path(self.X_tr, self.y_tr).ccp_alphas[:-1]
        ms = [DecisionTreeClassifier(ccp_alpha=a, random_state=1).fit(self.X_tr, self.y_tr) for a in al]
        return pd.DataFrame({'alpha': al, 'train': [m.score(self.X_tr, self.y_tr) for m in ms], 'test': [m.score(self.X_te, self.y_te) for m in ms]})

te_ = Tree_Experiment(Xb_tr, Xb_te, yb_tr, yb_te); print(te_.theo_do_sau().round(4).T)